# Exercise 5: Noise in search results

What if your topic model defines a cluster, but semantic search
returns "noisy" results that seem "off-topic". How to handle noisy datasets and
refine the embedding you use to search for topics. 

Goal: Understanding how to detect and address noise in real world datasets

In [ ]:
from pathlib import Path
import sys

sys.path.append("../")

# Check imports
from src.config import REPO

# Setup

In [ ]:
# Before we begin, we are going to add posts to our dataset so that it more
#closely mirrors a real world dataset. Run the "Setup" cell below, to update the
# sample data and retrain the topic model.

from pathlib import Path
import logging
import shutil
import sys
import json

sys.path.append("../")

# Check imports
from src.config import REPO
from solutions.data_models import PostDocument
from solutions.preprocess import PreprocessingPipeline as PreprocessingPipelineSolution

# To save time and ensure we have consistent data, we will load a file of sample
# data that contains additional randomly generated posts (i.e. "noise")

def load_preprocessed_data() -> list[PostDocument]:
    ## NOISE - load sample data with random posts.
    source_path = REPO / "solutions" / "processed_posts_noise.json.bkp"
    if not source_path.exists():
        raise FileNotFoundError(f"Missing source file: {source_path}")
    with open(source_path, "r") as f:
        data = json.load(f)
    raw_items = data.values()
    return [PostDocument(**item) for item in raw_items]

def run_truncated_pipeline():
    # Load preprocessed data (without vision features)
    postdocs = load_preprocessed_data()
    # Store results: clear existing outputs, then save to Elasticsearch (if
    # available) and disk.
    preprocesser = PreprocessingPipelineSolution()
    preprocesser.clear_stored_outputs()
    preprocesser.save_to_elasticsearch(postdocs)
    preprocesser.save_processed_posts(postdocs)

run_truncated_pipeline()

from solutions.topic_model import TopicModeler as TopicModelerSolution
TopicModelerSolution().run()

# Time: 2-10 minutes, depending on your machine.

## Outline

Exercise recap: What was precision@k? How about recall@k?
1. Diagnosing Noise
- Step 1: Add noise 
  - run the "Setup" cell above. It will load sample data that includes
    additional randomly generated posts to more closely replicate a real world
    data test. 
- Step 2 — Explore the model output

2. Localized vs Topic Embeddings
- Understanding the problem 
- Implementing "localized" embeddings

- Optional Coding Challenge
- Exercise: Evaluate topic results in the Demo App


# 1. Diagnosing Noise

### Step 1 — Add noise to the dataset and retrain

- Run the "Setup" cell above if you have not already. It will load sample data that includes
additional randomly generated posts to more closely replicate a real world
data test. 
- Remind me to ask you to take a quick survey, while that runs. 
- Once it finishes, reload the Demo App, open **Space Exploration**, and notice the
off-topic results that have crept in. We're going to diagnose them below.

### Step 2 — Explore the model output

After retraining, `output/` holds everything we need:

- `topic_information.csv` — size + label of every topic.
- `topic_assignments.csv` — which post is in which topic (and the dreaded `-1`).
- `topic_embeddings.json` — the **centroid** embedding per topic (mean of doc embeddings).
- `topic_keyword_embeddings.json` — the **localized** keyword embedding per topic (the
  embedding of just the top KeyBERT terms; computed in
  `src/topic_model.py:_save_keyword_embeddings`).

In [ ]:
import json

import numpy as np
import pandas as pd

from src.config import OUTPUT

topic_info = pd.read_csv(OUTPUT / "topic_information.csv")
assignments = pd.read_csv(OUTPUT / "topic_assignments.csv")
labels = json.loads((OUTPUT / "topic_labels.json").read_text())

print("Topic sizes (note size of -1 outlier bucket):")
print(topic_info[["Topic", "Count", "Name"]].head(15))

outliers = assignments[assignments["topic_id"] == -1]
pct = 100 * len(outliers) / len(assignments) if len(assignments) else 0
print(f"\nOutliers (topic_id == -1): {len(outliers)} of {len(assignments)} ({pct:.1f}%)")

print("\nFirst 5 outlier posts:")
for text in outliers["text"].head(5):
    print(f"  • {text[:120]}")

### Visual: How are the topics clustered? 

Open `output/topic_visualization.html` (the intertopic distance map). Each topic is
a circle in 2-D UMAP space; size is the number of docs assigned.  

- Are topics isolated or overlapping? 

# 2. Localized vs Topic Embeddings

## Understanding the problem

Even when BERTopic finds a coherent cluster (high c-TF-IDF, tight cosine distances
in the heatmap), running a *search* with the topic embedding can return obviously
off-topic posts.  

What's happening: the topic embedding is the **centroid** (mean) of every document
embedding in the cluster. When the cluster contains a diffuse, generalist
documents, that centroid expands — everything *kind of* matches everything.

<br><br>

<img src="images/topic_vs_localized_embedding.png" alt="Topic Embedding vs
Localized Embedding Illustration" style="width: 100%; max-width: 100%; height: auto;" />

## Topic embedding vs localized embedding

- **Topic Embedding** (mean of doc embeddings) 
    - The geometric centre ("centroid") of the cluster in 384-d space
    - Broadly represents the topic as a whole 
    - If the topic is broad, diffuse, the topic embedding representing it
      encompasses more area. 
- **Localized** (embedding of keywords or distinguishign features) 
    - An embedding generated from the most representative documents or keywords 
    - A more "localized" embedding that represents what makes the topic
      distinctive 
    - If the topic is broad, diffuse, a "localized" embedding can help separate
      posts that are "on topic" from more general posts. 

When the cluster contains posts with general, diffuse content, the topic
embeddings encompasses a larger "area". It matches more documents. 



## Generating a localized embedding

There are a few ways you can create an embedding that is tighter. 

- Use representative documents (ask an LLM to return a summary or keywords)
- Use top-n keywords from c-TF-IDF 
- Identify keywords in the topic that are the most representative of the topic
  because they are most similar to the topic embedding (KeyBERT)

  Here we will use KeyBERT to create topic embeddings that we can use for search.

### KeyBERT representations: the localized embedding source

BERTopic's `KeyBERTInspired` representation 
ranks candidate n-grams from a topic's documents by **cosine similarity to the topic
embedding**. Top-N of those are stored on
`topic_model.topic_aspects_["KeyBERT"]`.

TFI-DF captures the words that distinguish the topic from other topics. 

KeyBERT takes a sample of documents and candidate keywords and measures which is
semantically most meaningful to the topic. 

It does this by calculating the
similarity between candidate keywords and the topic using cosine similarity.
Then it returns the top-n most similar keywords. 


Reference: https://maartengr.github.io/BERTopic/getting_started/representation/representation.html

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic.load(str(OUTPUT / "bertopic_model"))

topic_info.head()

# Notice how the Representation kewords generated from TF-IDF are slightly
# different from the ones generated by KeyBERT.

In [ ]:
# Compare the topic embedding and the localized (keyword) embedding for one topic.
TOPIC_ID = 1 # case-insensitive substring

# Load the standard topic embeddings
topic_embeddings = json.loads((OUTPUT / "topic_embeddings.json").read_text())

# Load the localized topic embeddings (built from KeyBERT keywords)
keyword_embeddings = json.loads((OUTPUT / "topic_keyword_embeddings.json").read_text())

target_topic_id = TOPIC_ID

standard_topic_embedding = np.array(topic_embeddings[str(target_topic_id)], dtype=np.float32)
localized_topic_embedding = np.array(keyword_embeddings[str(target_topic_id)], dtype=np.float32)

cos = float(np.dot(standard_topic_embedding, localized_topic_embedding) /
            (np.linalg.norm(standard_topic_embedding) * np.linalg.norm(localized_topic_embedding)))
print(f"Topic {target_topic_id}: {labels[str(target_topic_id)]['label']}")
print(f"  cosine(centroid, localized) = {cos:.4f}")
print(f"  distance                    = {1 - cos:.4f}")
print("  → A larger distance means the keyword embedding is pulling the search")
print("    away from the centroid — useful when the centroid is diffuse.")

# Optional Coding Challenge: 
  Building on the code above, find the topic with the greatest difference (distance) between the topic
  embedding and the keyword embedding. 
  
  Which topic is it? 

  See reference implementation in ` solutions/challenge.py` for a solution

# Exercise 

Time: ~5-10 minutes

Once the pipeline has rerun and the model has been retrained in the [# Setup] cell
above, evaluate the results. 

1. Run `uv run streamlit run app.py` to start/restart the Demo App if it is not
   running. 
2. Open the [Demo App](http://127.0.0.1:8501) 

3. The demo app has a **"use 'localized' search embedding"** toggle. Pick the topic
you found above (or any topic with high centroid-to-localized distance) and flip it.

4. Review search results in the Demo App  
  -  Select the topic with the greatest difference (distance) between the topic
     embedding and the "localized" keyword embedding. 
  -  Toggle the `use "localized" search embedding` toggle. 
  - How do the results for the topics change? What happens to the search score? 
